In [1]:
import numpy as np

In [46]:
X = np.array([1,2,3]).astype(np.float32)

with open("logtest.spire", 'wb') as f:
    f.write(X.size.to_bytes(4,'big'))
    f.write(X.tobytes())


In [ ]:

f = open("logtest.spire", 'rb')
n_neurons = int.from_bytes(f.read(4), "big")
np.frombuffer(f.read(), dtype=np.float32).reshape(-1, n_neurons)




In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib
import bz2
import gzip
import lzma
from pathlib import Path

filename = "./cuda_test_9.spire.gz"
record_stride = 1  # set to the stride used when recording

def _open_spire_file(path: str):
    lower = path.lower()
    if lower.endswith(".spire.gz"):
        return gzip.open(path, "rb")
    if lower.endswith(".spire.xz") or lower.endswith(".spire.lzma"):
        return lzma.open(path, "rb")
    if lower.endswith(".spire.bz2"):
        return bz2.open(path, "rb")
    if lower.endswith(".spire"):
        return open(path, "rb")
    raise ValueError("Unsupported file type.")

def _spire_stem(path: str) -> str:
    name = Path(path).name
    lower = name.lower()
    for ext in (".spire.gz", ".spire.xz", ".spire.lzma", ".spire.bz2", ".spire"):
        if lower.endswith(ext):
            return name[: -len(ext)]
    return Path(path).stem

def play_spire_recording(filename: str, save = False, record_stride: int = 1):
    matplotlib.use('TkAgg')

    if filename == '':
        return

    if record_stride < 1:
        raise ValueError("record_stride must be >= 1.")

    try:
        f = _open_spire_file(filename)
    except ValueError:
        print("Cannot parse non-SpiRe file.")
        return

    try:
        with f:
            header = f.read(4)
            if len(header) < 4:
                print("Invalid or empty recording.")
                return
            neuron_count = int.from_bytes(header, byteorder='big')
            f32_bytesize = 4
            tickdata_size = neuron_count * f32_bytesize
            frames = []
            while True:
                tick_bytes = f.read(tickdata_size)
                if not tick_bytes:
                    break
                if len(tick_bytes) != tickdata_size:
                    print("Truncated recording.")
                    return
                frames.append(np.frombuffer(tick_bytes, dtype=np.float32, count=neuron_count))
            if not frames:
                print("Recording contains no frames.")
                return
            recording = np.vstack(frames)
            print(recording.shape)

    except Exception as e:
        print(e)
        return

    N = int(math.sqrt(neuron_count))
    fig, ax = plt.subplots()
    im = ax.imshow(recording[0].reshape(N, N), cmap='viridis', aspect='auto')
    plt.colorbar(im, ax=ax)
    ax.set_title(f'Tick : {0 * record_stride}')

    def update(frame):
        im.set_array(recording[frame].reshape(N, N))
        ax.set_title(f'Tick : {frame * record_stride}')
        return [im]

    anim = animation.FuncAnimation(
        fig, 
        update,
        frames=recording.shape[0],
        interval=1000/60,
        blit=True,
        repeat=False
    )

    if save: 
        out_stem = _spire_stem(filename)
        anim.save(f'{out_stem}.mp4', writer='ffmpeg', fps=60)
    else: 
        plt.show()


play_spire_recording(filename, save=False, record_stride=record_stride)





(4546, 65536)


In [ ]:
X.size